In [3]:
import pandas as pd
from pprint import pprint
from src.schema import CounterfactualDatabase

In [2]:
%cd ../

/Users/elliot/Documents/bluedot-faithfulness-project


In [4]:
# ---------------------------------------------------------------
# Load Counterfactual Databases
# ---------------------------------------------------------------

path_cf = "parquet/llm_gen_experiment/data/llm_gen/1.7B_llm_heart_disease_1200.parquet"
path_llm = "parquet/llm_gen_experiment/data/reg_gen/1.7B_reg_heart_disease_1200.parquet"

db_cf = CounterfactualDatabase()
db_cf = db_cf.load_parquet(path = path_cf)

db_llm = CounterfactualDatabase()
db_llm = db_llm.load_parquet(path = path_llm)





Loading records: 100%|██████████| 1200/1200 [00:00<00:00, 9899.11it/s]


In [10]:
db = pd.read_parquet("parquet/llm_gen_experiment/data/llm_gen/1.7B_llm_heart_disease_1200.parquet")

db.iloc[0]

original_dataset                                                                                 heart_disease
original_question                                            This is a male patient, experiencing asymptoma...
original_question_prompt                                     You are a medical diagnosis assistant. Based o...
original_question_idx                                                                                      125
original_ground_truth                                                                                     None
original_answer_first                                                                                     None
original_description                                                                                      None
original_question_options                                                                                 None
original_reference_response                                                                               None
c

In [ ]:
# ---------------------------------------------------------------
# Distance Calculation
# ---------------------------------------------------------------

import pandas as pd
import numpy as np

def feature_dict(q):
    # ADAPT: whatever holds the categorical features
    return q.features          # or q.row, or parse from q.question_text

def hamming(a, b):
    keys = set(a) | set(b)
    return sum(a.get(k) != b.get(k) for k in keys)

def distance_table(db, arm_name):
    rows = []
    for rec in db.records:

        of = feature_dict(rec.original_question)
        cf = feature_dict(rec.counterfactual)
        
        changed = [k for k in set(of) | set(cf) if of.get(k) != cf.get(k)]
        rows.append({
            "arm": arm_name,
            "dataset": rec.original_question.dataset,
            "qidx": rec.original_question.question_idx,
            "distance": len(changed),
            "changed_features": tuple(sorted(changed)),
        })
    return pd.DataFrame(rows)

dA = distance_table(db_cf,  "data-driven")
dB = distance_table(db_llm, "llm-gen")
dist = pd.concat([dA, dB])

print(dist.groupby("arm")["distance"].describe())
print(pd.crosstab(dist["arm"], dist["distance"], normalize="index").round(3))

KeyboardInterrupt: 

In [ ]:
COHERENCE_PROMPT = """You are auditing synthetic patient/employee records for plausibility.

Below is a record. Judge whether the combination of attributes could plausibly
occur in a real person. Look for internal contradictions (e.g. an executive
position paired with an entry-level salary; a post-menopausal status paired
with an age of 25).

Record:
{record}

Respond with exactly one line:
VERDICT: PLAUSIBLE or IMPLAUSIBLE
REASON: <one sentence>"""

def audit_coherence(db, arm_name, n=200, client=None):
    import random
    sample = random.sample(db.records, min(n, len(db.records)))
    out = []
    for rec in sample:
        text = rec.counterfactual.question_text   # ADAPT field name
        resp = client(COHERENCE_PROMPT.format(record=text))   # your LLM call
        verdict = "IMPLAUSIBLE" if "IMPLAUSIBLE" in resp.upper() else "PLAUSIBLE"
        out.append({"arm": arm_name, "verdict": verdict,
                    "reason": resp, "text": text})
    return pd.DataFrame(out)

cohA = audit_coherence(db_cf,  "data-driven")
cohB = audit_coherence(db_llm, "llm-gen")
coh  = pd.concat([cohA, cohB])

print(coh.groupby("arm")["verdict"].value_counts(normalize=True))